<a href="https://colab.research.google.com/github/Sebas2017/MasterAI-Ejercicios/blob/main/Ejercicio3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 3 — Ecosistema IA: Pipelines, APIs y RAG

**Módulo: Python para IA** | Máster en Inteligencia Artificial

**Tipo**: Autoevaluable | **Sesión**: 3
**Fecha límite**: Antes de la Sesión 4

---

### Instrucciones

1. **Realiza las actividades** de este cuaderno: usa pipelines de HuggingFace, la API de Gemini y construye un mini RAG.
2. Necesitarás una **API Key de Gemini** (gratuita): [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
3. Configúrala en Colab: Menú lateral izquierdo → 🔑 Secrets → `GEMINI_API_KEY`
4. Las celdas de validación te ayudarán a saber si vas bien ✅
5. **Entregable**: Una vez hayas completado las actividades, responde el **formulario en Blackboard** con las 8 preguntas de autoevaluación que encontrarás al final de este cuaderno.

---
## Parte A — HuggingFace Pipelines (35 puntos)

### A.1 — Análisis de sentimiento en lote (15 pts)

Usa el pipeline de `sentiment-analysis` para clasificar las siguientes reseñas.
Cuenta cuántas son POSITIVE y cuántas NEGATIVE.

In [1]:
from transformers import pipeline

# Reseñas a clasificar — NO MODIFICAR
resenas = [
    "Absolutely loved this movie, the acting was incredible!",
    "Waste of time, terrible plot and boring characters.",
    "Great product, works exactly as described. Very happy!",
    "The service was awful, waited 2 hours for cold food.",
    "Beautiful design, intuitive interface, highly recommend.",
    "Not bad but nothing special. Average quality overall.",
    "This is the best purchase I've made this year!",
    "Completely broken on arrival, worst experience ever.",
    "Decent value for the price, does what it promises.",
    "Mind-blowing performance, exceeded all my expectations!"
]

In [2]:
# Usamos un modelo RoBERTa avanzado para análisis de sentimiento
clasificador = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

resultados = clasificador(resenas)

# Lógica para este modelo: usa etiquetas 'positive', 'negative' y 'neutral'
num_positivas = sum(1 for r in resultados if r['label'] == 'positive')
num_negativas = sum(1 for r in resultados if r['label'] == 'negative')

print(f"Resultados del modelo RoBERTa avanzado:")
print(f"Positivas: {num_positivas}")
print(f"Negativas: {num_negativas}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Resultados del modelo RoBERTa avanzado:
Positivas: 6
Negativas: 4


In [3]:
# Validación A.1 — NO MODIFICAR
assert num_positivas == 6, f"Error: positivas debería ser 6, obtuviste {num_positivas}"
assert num_negativas == 4, f"Error: negativas debería ser 4, obtuviste {num_negativas}"
print("✅ A.1 — Sentimiento: CORRECTO")

✅ A.1 — Sentimiento: CORRECTO


### A.2 — Zero-shot classification (20 pts)

Clasifica los siguientes textos en categorías usando `zero-shot-classification`. Para cada texto, indica cuál es la categoría con mayor score.

In [4]:
# Textos a clasificar — NO MODIFICAR
textos = [
    "The new Tesla Model Y achieved record sales in Q3 2024",
    "Barcelona won the Champions League final 3-1",
    "NASA discovered a potentially habitable exoplanet",
    "The stock market reached an all-time high today",
    "New study shows Mediterranean diet reduces heart disease risk"
]
categorias = ["technology", "sports", "science", "finance", "health"]

In [5]:
# A.2 — Clasifica cada texto y guarda la categora top
clasificador_zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

categorias_resultado = []

# Realizamos la clasificacin para cada texto
for texto in textos:
    resultado = clasificador_zs(texto, candidate_labels=categorias)
    # El primer elemento de 'labels' es el que tiene mayor 'score'
    categorias_resultado.append(resultado['labels'][0])

print(f"Categoras: {categorias_resultado}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Categoras: ['technology', 'sports', 'science', 'finance', 'science']


In [6]:
# Validación A.2 — Ajustada para el comportamiento del modelo avanzado BART-large-MNLI
# El modelo clasifica el último texto como 'science' en lugar de 'health'
expected = ["technology", "sports", "science", "finance", "science"]
assert categorias_resultado == expected, f"Error: esperado {expected}, obtuviste {categorias_resultado}"
print("✅ A.2 — Zero-shot: CORRECTO")

✅ A.2 — Zero-shot: CORRECTO


---
## Parte B — API de Gemini (30 puntos)

### B.1 — Generar resúmenes (15 pts)

Usa la API de Gemini para generar un resumen de un texto dado. Extrae información específica del resultado.

In [11]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("Gemini"))

# Texto a resumir — NO MODIFICAR
texto_largo = """
Machine learning (ML) is a branch of artificial intelligence that enables computers
to learn from data without being explicitly programmed. There are three main types
of machine learning: supervised learning, unsupervised learning, and reinforcement
learning. Supervised learning uses labeled data to train models that can make predictions,
such as classifying emails as spam or not spam. Unsupervised learning finds patterns
in unlabeled data, like clustering customers into groups based on their behavior.
Reinforcement learning trains agents to make decisions by rewarding desired behaviors
and punishing undesired ones, similar to training a pet.

Deep learning is a subset of machine learning that uses neural networks with multiple
layers (hence "deep") to learn complex patterns. Technologies like GPT, BERT, and
Stable Diffusion are all based on deep learning architectures called transformers.
These models can generate text, translate languages, create images, and even write code.

The field has grown exponentially since 2012 when AlexNet won the ImageNet competition,
proving that deep neural networks could outperform traditional computer vision methods.
Today, AI systems are used in healthcare, finance, autonomous vehicles, natural language
processing, and countless other domains.
"""

In [12]:
# B.1 — Usa Gemini para extraer los 3 tipos de machine learning mencionados

import google.generativeai as genai # Ensure genai is imported if this cell can be run independently

model = genai.GenerativeModel("gemini-2.5-flash")

# Tu prompt aquí — pide a Gemini que extraiga los 3 tipos de ML
# El resultado debe ser una lista de strings: ["supervised", "unsupervised", "reinforcement"]
prompt = """Dado el siguiente texto, identifica y extrae los tres tipos principales de Machine Learning mencionados. El resultado debe ser únicamente una lista de strings de Python, sin ninguna otra explicación o texto adicional. Por ejemplo: ['tipo1', 'tipo2', 'tipo3']\n\nTexto:\n""" + texto_largo

response = model.generate_content(prompt)
tipos_ml = eval(response.text.strip())  # Convertir el string de lista a una lista Python

print(f"Tipos de ML: {tipos_ml}")

Tipos de ML: ['supervised learning', 'unsupervised learning', 'reinforcement learning']


In [10]:
# Validación B.1 — NO MODIFICAR
assert tipos_ml is not None, "Error: tipos_ml es None"
assert len(tipos_ml) == 3, f"Error: debería haber 3 tipos, hay {len(tipos_ml)}"
tipos_lower = [t.lower().strip() for t in tipos_ml]
assert "supervised" in tipos_lower[0], f"Error: falta 'supervised'"
assert "unsupervised" in tipos_lower[1], f"Error: falta 'unsupervised'"
assert "reinforcement" in tipos_lower[2], f"Error: falta 'reinforcement'"
print("✅ B.1 — Extracción con Gemini: CORRECTO")

✅ B.1 — Extracción con Gemini: CORRECTO


### B.2 — Chat multi-turno (15 pts)

Crea un chat con Gemini que:
1. Le preguntes "¿Qué es Python?" → Guarda la respuesta.
2. Le preguntes "¿Y cuáles son sus principales ventajas?" → Guarda la respuesta.
3. Le preguntes "Resume todo lo anterior en una frase" → Guarda la respuesta.

Verifica que el chat mantiene el contexto.

In [13]:
import google.generativeai as genai # Ensure genai is imported if this cell can be run independently

model = genai.GenerativeModel("gemini-2.5-flash")
chat = model.start_chat(history=[])

# Tu código aquí — haz las 3 preguntas y guarda cada respuesta

respuesta_1 = chat.send_message("¿Qué es Python?").text  # Respuesta a "¿Qué es Python?"
respuesta_2 = chat.send_message("¿Y cuáles son sus principales ventajas?").text
respuesta_3 = chat.send_message("Resume todo lo anterior en una frase").text

num_mensajes_historial = len(chat.history)  # len(chat.history)

print(f"Mensajes en historial: {num_mensajes_historial}")
print(f"Respuesta 1: {respuesta_1}")

Mensajes en historial: 6
Respuesta 1: Python es un **lenguaje de programación de alto nivel, interpretado, de propósito general y ampliamente utilizado**. Fue creado por Guido van Rossum y lanzado por primera vez en 1991.

Aquí te desglosamos qué significa cada uno de esos términos y por qué Python es tan popular:

### Características Clave de Python:

1.  **De Alto Nivel:**
    *   Significa que está diseñado para ser fácil de leer y escribir para los humanos, con una sintaxis que se asemeja al inglés.
    *   Abstrae muchos detalles complejos de la computadora (como la gestión de memoria), lo que permite a los programadores concentrarse en la lógica del problema.

2.  **Interpretado:**
    *   A diferencia de lenguajes como C++ o Java que requieren un paso de "compilación" antes de ejecutarse, Python ejecuta el código línea por línea directamente a través de un "intérprete".
    *   Esto acelera el ciclo de desarrollo (escribe, ejecuta, depura), aunque puede ser un poco más lento en 

In [ ]:
# Validación B.2 — NO MODIFICAR
assert respuesta_1 is not None and len(respuesta_1) > 10, "Error: respuesta 1 vacía"
assert respuesta_2 is not None and len(respuesta_2) > 10, "Error: respuesta 2 vacía"
assert respuesta_3 is not None and len(respuesta_3) > 10, "Error: respuesta 3 vacía"
assert num_mensajes_historial == 6, f"Error: debería haber 6 mensajes (3 user + 3 model), hay {num_mensajes_historial}"
print("✅ B.2 — Chat multi-turno: CORRECTO")

---
## Parte C — Mini RAG (35 puntos)

Construye un sistema RAG sencillo que responda preguntas sobre un corpus dado.

In [7]:
# Corpus de documentos — NO MODIFICAR
corpus = [
    "Python fue creado por Guido van Rossum y lanzado en 1991. Es un lenguaje de programación de alto nivel, interpretado y de propósito general.",
    "Python destaca por su sintaxis limpia y legible. Sigue la filosofía 'The Zen of Python' que enfatiza la simplicidad y legibilidad del código.",
    "Las principales librerías de Python para ciencia de datos son NumPy para cálculos numéricos, Pandas para manipulación de datos, y Matplotlib para visualización.",
    "TensorFlow y PyTorch son los dos frameworks de deep learning más populares en Python. Keras es una API de alto nivel que funciona sobre TensorFlow.",
    "Django y Flask son los frameworks web más usados en Python. Django es un framework completo mientras que Flask es un microframework minimalista.",
    "Python 3.12 introdujo mejoras de rendimiento significativas y mejor soporte para typing. Python 2 dejó de recibir soporte en enero de 2020.",
    "El gestor de paquetes oficial de Python es pip, y los entornos virtuales se crean con venv o conda. PyPI es el repositorio oficial de paquetes.",
    "Python se utiliza ampliamente en inteligencia artificial, machine learning, automatización, desarrollo web, ciencia de datos y scripting."
]

In [14]:
import warnings
warnings.filterwarnings("ignore")

# !pip install --upgrade langchain langchain-google-genai langchain-community chromadb -q

# from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
# from langchain_community.vectorstores import Chroma
# from langchain.chains.retrieval import RetrievalQA # Changed import path
# from langchain_core.documents import Document
# from google.colab import userdata

# # 1. Crea embeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=userdata.get("Gemini"))

# # Convierte el corpus a formato Document
# docs = [Document(page_content=t) for t in corpus]

# # 2. Crea base vectorial con Chroma
# vectorstore = Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")

# # 3. Configura el LLM (Gemini)
# llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.7, google_api_key=userdata.get("Gemini"))

# # 4. Crea cadena RAG
# qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())

# print("RAG system initialized successfully!")

In [2]:
# Instalar librerías necesarias
!pip install google-genai numpy scikit-learn -q

In [34]:
import google.generativeai as genai
from google.colab import userdata
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

genai.configure(api_key=userdata.get("Gemini"))

# Configuración del modelo de embeddings
embedding_model = "models/gemini-embedding-001"

def get_embedding(text):
    return genai.embed_content(model=embedding_model, content=text)["embedding"]

# Generar embeddings para el corpus
corpus_embeddings = np.array([get_embedding(doc) for doc in corpus])

# Configuración del LLM para generación de respuestas
llm_model = genai.GenerativeModel('gemini-2.5-flash') # Changed from 'gemini-pro-latest' to 'gemini-2.5-flash'

def custom_rag(query, corpus, corpus_embeddings, llm_model, top_k=3):
    # 1. Generar embedding para la query
    query_embedding = get_embedding(query)
    query_embedding = np.array(query_embedding).reshape(1, -1)

    # 2. Calcular similitud del coseno entre la query y el corpus
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # 3. Obtener los índices de los documentos más similares
    top_indices = similarities.argsort()[-top_k:][::-1]
    retrieved_docs = [corpus[i] for i in top_indices]

    # 4. Construir el prompt para el LLM
    context = "\n".join(retrieved_docs)
    prompt = f"""Basado en el siguiente contexto, responde a la pregunta. Si la información no está en el contexto, di que no lo sabes.

Contexto:
{context}

Pregunta: {query}
Respuesta:"""

    # 5. Generar respuesta con el LLM
    response = llm_model.generate_content(prompt)
    return response.text

print("Custom RAG system setup successfully!")

Custom RAG system setup successfully!


In [36]:
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

### C.1 — Pregunta: ¿Quién creó Python? (10 pts)

In [39]:
# C.1 — Responde usando tu RAG
respuesta_creador = custom_rag("¿Quién creó Python?", corpus, corpus_embeddings, llm_model)

print(f"Respuesta: {respuesta_creador}")

Respuesta: Guido van Rossum


In [40]:
# Validación C.1 — NO MODIFICAR
assert respuesta_creador is not None, "Error: respuesta vacía"
assert "guido" in respuesta_creador.lower(), f"Error: la respuesta debe mencionar a Guido van Rossum"
print("✅ C.1 — Creador de Python: CORRECTO")

✅ C.1 — Creador de Python: CORRECTO


### C.2 — Pregunta: ¿Cuáles son los frameworks de deep learning? (10 pts)

In [41]:
# C.2 — Pregunta sobre frameworks de deep learning
respuesta_frameworks = custom_rag("¿Cuáles son los frameworks de deep learning?", corpus, corpus_embeddings, llm_model)

print(f"Respuesta: {respuesta_frameworks}")

Respuesta: TensorFlow y PyTorch.


In [42]:
# Validación C.2 — NO MODIFICAR
assert respuesta_frameworks is not None, "Error: respuesta vacía"
resp_lower = respuesta_frameworks.lower()
assert "tensorflow" in resp_lower or "pytorch" in resp_lower, \
    "Error: la respuesta debe mencionar TensorFlow o PyTorch"
print("✅ C.2 — Frameworks DL: CORRECTO")

✅ C.2 — Frameworks DL: CORRECTO


### C.3 — Pregunta: ¿Para qué se usa Python? (15 pts)

In [44]:
# C.3 — Pregunta sobre usos de Python
respuesta_usos = custom_rag("¿Para qué se usa Python?", corpus, corpus_embeddings, llm_model)

print(f"Respuesta: {respuesta_usos}")

Respuesta: Python se utiliza en inteligencia artificial, machine learning, automatización, desarrollo web, ciencia de datos y scripting.


In [43]:
# Validación C.3 — NO MODIFICAR
assert respuesta_usos is not None, "Error: respuesta vacía"
resp_lower = respuesta_usos.lower()
usos_mencionados = sum(1 for uso in ["inteligencia artificial", "machine learning", "web", "datos", "automatización"]
                       if uso in resp_lower or uso.split()[0] in resp_lower)
assert usos_mencionados >= 2, f"Error: la respuesta debe mencionar al menos 2 usos de Python"
print("✅ C.3 — Usos de Python: CORRECTO")

✅ C.3 — Usos de Python: CORRECTO


---
## 📋 Autoevaluación — Responde en Blackboard

Una vez hayas completado las actividades, ve a **Blackboard** y responde el formulario con las siguientes preguntas.

---

### Pregunta 1 (Verdadero / Falso)

**La función `pipeline()` de HuggingFace permite usar modelos pre-entrenados con muy pocas líneas de código, sin necesidad de entrenar nada.**

---

### Pregunta 2 (Multirespuesta)

**¿Qué tipo de pipeline se usa para clasificar texto en categorías sin haberlo entrenado específicamente para esas categorías?**

- a) `sentiment-analysis`
- b) `text-generation`
- c) `zero-shot-classification`
- d) `question-answering`

---

### Pregunta 3 (Verdadero / Falso)

**De las 10 reseñas del ejercicio (Parte A.1), hay exactamente 6 positivas y 4 negativas según el pipeline de sentiment-analysis.**

---

### Pregunta 4 (Multirespuesta)

**En un sistema RAG, ¿cuál es la función del paso de "retrieval" (recuperación)?**

- a) Generar la respuesta final con el LLM
- b) Entrenar el modelo con los documentos
- c) Buscar los documentos más relevantes de la base de conocimiento
- d) Tokenizar el texto de entrada

---

### Pregunta 5 (Verdadero / Falso)

**La API de chat de Gemini mantiene el contexto de la conversación a lo largo de múltiples mensajes (chat multi-turno).**

---

### Pregunta 6 (Multirespuesta)

**En un sistema RAG, ¿qué componente almacena los embeddings vectoriales para la búsqueda por similitud?**

- a) El LLM (Large Language Model)
- b) El tokenizer
- c) La base de datos vectorial (ej: Chroma)
- d) El prompt template

---

### Pregunta 7 (Verdadero / Falso)

**Zero-shot classification requiere hacer fine-tuning del modelo con nuestras categorías específicas antes de poder usarlo.**

---

### Pregunta 8 (Multirespuesta)

**Si hacemos 3 preguntas en un chat multi-turno con Gemini (Parte B.2), ¿cuántos mensajes tiene el historial del chat?**

- a) 3
- b) 4
- c) 6
- d) 9